# Pretraining integral — LLM Legislativo Brasileiro

Este fluxo consome a release curada unificada, os shards completos e persiste checkpoints no Google Drive. O treino termina após uma passagem determinística por todas as janelas do split `train`; não reduz o corpus a um smoke. A cobertura histórica da release segue parcial.

Pré-requisitos: copie `code-integral-20261011.zip` para `MyDrive/llm_legislativo/code/`, extraia o conteúdo em `code/current/` (incluindo `bundle_manifest.json` e `BUNDLE_COMPLETE.json`) e copie a release `corpus-legislativo-current-v1`, copie a release para `releases/curated/corpus-legislativo-current-v1`, o tokenizer local `artifacts/tokenizer/legislativo-atual-v1` para `releases/tokenizers/legislativo-atual-v1` e os shards locais `artifacts/tokens/legislativo-atual-v1` para `releases/tokens/legislativo-atual-v1`. A conta do Colab deve ser a conta Google pessoal escolhida pelo pesquisador. Este notebook não envia dados do computador para o Drive.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, math, shutil, yaml
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/llm_legislativo')
CODE_ROOT = Path(os.environ.get('LLM_CODE_ROOT', DRIVE_ROOT / 'code/current'))
RELEASE_DIR = Path(os.environ.get('LLM_RELEASE_DIR', DRIVE_ROOT / 'releases/curated/corpus-legislativo-current-v1'))
TOKEN_SOURCE = Path(os.environ.get('LLM_TOKEN_SOURCE_DIR', DRIVE_ROOT / 'releases/tokens/legislativo-atual-v1'))
TOKENIZER_DIR = Path(os.environ.get('LLM_TOKENIZER_DIR', DRIVE_ROOT / 'releases/tokenizers/legislativo-atual-v1'))
if not (CODE_ROOT / 'src/legislm').is_dir(): raise FileNotFoundError(f'Bundle legislm ausente: {CODE_ROOT}')
bundle_manifest_path = CODE_ROOT / 'bundle_manifest.json'
bundle_complete_path = CODE_ROOT / 'BUNDLE_COMPLETE.json'
bundle_manifest_sha = hashlib.sha256(bundle_manifest_path.read_bytes()).hexdigest()
bundle_complete = json.loads(bundle_complete_path.read_text())
if bundle_complete.get('manifest_sha256') != bundle_manifest_sha: raise RuntimeError('Bundle de código incompleto ou alterado')
expected_bundle_sha = os.environ.get('LLM_CODE_BUNDLE_SHA256')
if expected_bundle_sha and expected_bundle_sha != bundle_manifest_sha: raise RuntimeError('Hash do bundle de código diverge do valor fixado')
for entry in json.loads(bundle_manifest_path.read_text())['files']:
    f = CODE_ROOT / entry['path']
    if not f.is_file() or f.stat().st_size != entry['size_bytes'] or hashlib.sha256(f.read_bytes()).hexdigest() != entry['sha256']:
        raise RuntimeError(f'Arquivo do bundle de código inválido: {f}')
print('Bundle de código:', bundle_manifest_sha)
REPO_ROOT = CODE_ROOT
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT / 'src'))
os.environ['DATA_ROOT'] = '/content/llm_legislativo/data'
Path(os.environ['DATA_ROOT']).mkdir(parents=True, exist_ok=True)
import torch, legislm
print('legislm', legislm.__version__, '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())


In [ ]:
from legislm.tokenization.reader import CuratedReleaseReader
from legislm.colab.preflight import run_preflight

# Confere o manifesto e os hashes de todos os arquivos listados na release curada.
release_manifest = CuratedReleaseReader(RELEASE_DIR).validate_release_integrity()
if release_manifest['coverage_status'] != 'partial':
    raise RuntimeError('Este notebook espera que a cobertura parcial esteja declarada no manifesto.')

def verify_manifest_files(root: Path, manifest_name: str):
    manifest = json.loads((root / manifest_name).read_text())
    for entry in manifest['files']:
        p = root / entry['path']
        if not p.is_file() or p.stat().st_size != entry['size_bytes']:
            raise RuntimeError(f'Arquivo ausente/incompleto: {p}')
        h = hashlib.sha256()
        with p.open('rb') as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(chunk)
        if h.hexdigest() != entry['sha256']: raise RuntimeError(f'SHA-256 divergente: {p}')
    return manifest

token_manifest = verify_manifest_files(TOKEN_SOURCE, 'token_dataset_manifest.json')
token_complete = json.loads((TOKEN_SOURCE / 'COMPLETE.json').read_text())
manifest_sha = hashlib.sha256((TOKEN_SOURCE / 'token_dataset_manifest.json').read_bytes()).hexdigest()
if token_complete.get('manifest_sha256') != manifest_sha: raise RuntimeError('COMPLETE do token dataset não corresponde ao manifesto')
if token_manifest['token_dataset_id'] != 'tokens_legislativo_atual_v1': raise RuntimeError('Token dataset inesperado')
preflight = run_preflight('configs/colab/integral.yaml', allow_cpu_fallback=False)
if preflight['status'] != 'PASSED': raise RuntimeError(f'Preflight G5 falhou: {preflight}')
print('Release, tokenizer/shards e preflight validados.')


In [ ]:
# Estagia o token dataset inteiro no disco efêmero local; os shards são hash-verificados.
from legislm.colab.staging import stage_file
TOKEN_LOCAL = Path(os.environ['DATA_ROOT']) / 'artifacts/tokens/tokens_legislativo_atual_v1'
TOKEN_LOCAL.mkdir(parents=True, exist_ok=True)
for entry in token_manifest['files']:
    src = TOKEN_SOURCE / entry['path']
    dst = TOKEN_LOCAL / entry['path']
    dst.parent.mkdir(parents=True, exist_ok=True)
    stage_file(src, dst, verify_hashes=True, force_overwrite=True)
for name in ('token_dataset_manifest.json', 'COMPLETE.json'):
    shutil.copy2(TOKEN_SOURCE / name, TOKEN_LOCAL / name)
for name in ('tokenizer.json', 'tokenizer_manifest.json', 'COMPLETE.json'):
    if not (TOKENIZER_DIR / name).is_file(): raise FileNotFoundError(TOKENIZER_DIR / name)


In [ ]:
# Dimensiona o scheduler para uma passagem completa pelas janelas train.
from legislm.tokenization.dataset import TokenDataset
from legislm.training.config import TrainingConfig
from legislm.training.engine import Trainer
from legislm.training.persistence import get_persistence_backend

train_data = TokenDataset.from_dataset_dir(TOKEN_LOCAL, split='train')
validation_data = TokenDataset.from_dataset_dir(TOKEN_LOCAL, split='validation')
if not len(train_data) or not len(validation_data): raise RuntimeError('Splits tokenizados vazios')
with (TOKENIZER_DIR / 'tokenizer_manifest.json').open() as f: tokenizer_manifest = json.load(f)

cfg = TrainingConfig.from_yaml(CODE_ROOT / 'configs/training/integral.yaml')
cfg.device = 'cuda'
cfg.model.vocab_size = int(tokenizer_manifest['vocab_size'])
effective_windows_per_update = cfg.training.micro_batch_size * cfg.training.grad_accum_steps
cfg.training.max_steps = math.ceil(len(train_data) / effective_windows_per_update)
cfg.training.warmup_steps = min(2000, max(1, round(cfg.training.max_steps * 0.03)))
cfg.training.evaluation_interval = cfg.training.max_steps  # avaliação completa única ao fim da passagem
cfg.training.checkpoint_interval_steps = cfg.training.max_steps  # checkpoints intermediários seguem o relógio de 5 min
cfg.persistence.runs_dir = str(DRIVE_ROOT / 'runs')
cfg.persistence.drive_root = str(DRIVE_ROOT)
cfg.validate()
run_root = DRIVE_ROOT / 'runs' / cfg.run_id
run_root.mkdir(parents=True, exist_ok=True)
resolved = cfg.to_dict()
(run_root / 'config.resolved.json').write_text(json.dumps(resolved, indent=2) + '\n')
print(f'train windows={len(train_data):,}; validation windows={len(validation_data):,}; updates for one pass={cfg.training.max_steps:,}')


In [ ]:
# Cada sessão continua até o fim do plano; em caso de interrupção, retome em uma sessão nova
# definindo LLM_RESUME_SELECTOR=latest depois de validar os checkpoints persistidos.
backend = get_persistence_backend('drive', checkpoints_to_keep=cfg.persistence.checkpoints_to_keep)
trainer = Trainer(cfg, train_dataset=train_data, eval_dataset=validation_data, persistence_backend=backend)
resume_selector = os.environ.get('LLM_RESUME_SELECTOR')
if resume_selector: trainer.resume_from_checkpoint(selector=resume_selector)
summary = trainer.run()
(run_root / 'training_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
summary
